<a href="https://colab.research.google.com/github/LeandroLDA/ColabNotebooks/blob/main/Olist_E_commerce_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Configuração base
sns.set_style("whitegrid")

print("⏳ Baixando dataset da Olist do Kaggle...")
try:
    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print(f"✅ Arquivos baixados em: {path}")

    # Carregando apenas 3 tabelas essenciais para o nosso escopo
    orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
    order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
    products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))

    print("✅ DataFrames 'orders', 'order_items' e 'products' carregados com sucesso!")
except Exception as e:
    print(f"❌ Erro ao ler arquivos: {e}")

⏳ Baixando dataset da Olist do Kaggle...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
✅ Arquivos baixados em: /kaggle/input/brazilian-ecommerce
✅ DataFrames 'orders', 'order_items' e 'products' carregados com sucesso!


In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [ ]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [ ]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


#📦 Desafio de Negócios: Olist E-commerce Analytics.
Contexto: Você é o novo Cientista de Dados da Olist. Você tem acesso a três bases de dados:

- ```orders```: Status do pedido e datas.
- ```order_items```: O que foi comprado, preço (price) e valor do frete (freight_value).
- ```products```: Categoria do produto e peso físico (product_weight_g).

## Requisições da Diretoria:
O board de diretores enviou 5 requisições de negócios.

### 1. Consolidação da Base (Data Engineering).

- A Demanda: A equipe de logística precisa de uma visão única (tabela mestra) que mostre as informações do pedido, os itens contidos nele e as características físicas desse produto (categoria e peso).
- A Regra: Queremos analisar exclusivamente os pedidos que possuam itens registrados e cujos produtos existam no nosso catálogo. Pedidos vazios ou produtos órfãos devem ficar de fora. Crie essa tabela e chame-a de ```df_master```.

### 2. Regra de Negócio de Frete (Tooling & Robustness).

- A Demanda: O financeiro quer calcular uma 'Taxa de Risco' para os fretes. A regra matemática é simples:</br></br>
 $$Taxa = Valor do Frete / Preço do Produto$$</br>
- A Regra: Crie uma ferramenta (função) reutilizável para este cálculo. No entanto, o sistema antigo às vezes exportava letras no lugar de números. A sua ferramenta deve ser inteligente o suficiente para não quebrar o código quando encontrar um texto, retornando 0 nestes casos. Além disso, a ferramenta deve bloquear ativamente (levantar um erro claro) caso receba um preço de produto negativo.

### 3. Aplicação da Regra (Data Transformation).

- A Demanda: Aplique a calculadora de 'Taxa de Risco' na nossa base consolidada (```df_master```), criando uma nova coluna com este resultado.
- A Regra: Utilize a abordagem mais otimizada e idiomática do Python para aplicar essa regra linha a linha, sem usar loops ```for``` tradicionais.

4. Gerador de Relatórios (Analytics).

- A Demanda: A diretoria precisa extrair métricas rápidas de preços por categoria de produto sem ter que reescrever código toda vez.
- A Regra: Construa um gerador (função) que receba a base de dados consolidada e o nome de uma categoria específica. Ele deve processar esses dados e entregar, de uma só vez, duas métricas sobre a coluna ```price``` dessa categoria: a média e a mediana. Teste o seu gerador solicitando os dados da categoria ```beleza_saude``` e exiba os resultados na tela.

### 5. Apresentação Executiva (Data Visualization).

- A Demanda: "Temos uma reunião de investidores amanhã. Precisamos de um gráfico que responda visualmente a duas perguntas ao mesmo tempo: Produtos mais pesados têm fretes mais caros? E como o preço do produto interage com isso?"
- A Regra: Crie a visualização apropriada para cruzar Peso, Frete e Preço. Sabendo que o gráfico será projetado numa tela grande de sala de reunião, garanta que os elementos visuais (tamanho da fonte, paleta de cores) estejam otimizados para apresentação, e não para a tela de um notebook comum. O gráfico deve ser autoexplicativo (títulos e eixos).